# Notebook 2 - Analyse des datasets

Ce notebook montre comment charger deux benchmarks de bon sens avec `datasets` :
- CommonsenseQA
- PIQA

L’objectif est d’ouvrir les données, vérifier leur schéma et repérer les éléments utiles pour la comparaison baseline / graphe.

In [1]:
from datasets import load_dataset

csqa_examples = [
    {
        "question": "Where would you find a refrigerator?",
        "choices": {"text": ["kitchen", "forest", "ocean", "school"]},
        "answerKey": "A",
    },
    {
        "question": "What do you usually use to drink water?",
        "choices": {"text": ["cup", "book", "pencil", "window"]},
        "answerKey": "A",
    },
]

piqa_examples = [
    {
        "goal": "How to make coffee?",
        "sol1": "Pour hot water on coffee grounds.",
        "sol2": "Put the coffee in the freezer.",
        "label": 0,
    },
    {
        "goal": "How to clean a table?",
        "sol1": "Use a damp cloth and wipe it.",
        "sol2": "Use a hammer on it.",
        "label": 0,
    },
]

try:
    csqa = load_dataset("commonsense_qa", split="train[:3]")
    piqa = load_dataset("piqa", split="train[:3]")
    print("Datasets chargés depuis Hugging Face.")
except Exception as exc:
    print("Chargement HF impossible :", exc)
    print("Utilisation d’un petit jeu de données local de démonstration.")
    csqa = csqa_examples
    piqa = piqa_examples

print("CommonsenseQA sample:")
print(csqa)
print("\nPIQA sample:")
print(piqa)


/home/timo27/epita/ing2/ia-symbo/2026-Epita-Intelligence-Symbolique/O3-Raisonnement-de-bon-sens/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Chargement HF impossible : Invalid HF URI 'hf://datasets/commonsense_qa@94630fe30dad47192a8546eb75f094926d47e155/.huggingface.yaml'. Repository id must be 'namespace/name', got 'commonsense_qa'.
Utilisation d’un petit jeu de données local de démonstration.
CommonsenseQA sample:
[{'question': 'Where would you find a refrigerator?', 'choices': {'text': ['kitchen', 'forest', 'ocean', 'school']}, 'answerKey': 'A'}, {'question': 'What do you usually use to drink water?', 'choices': {'text': ['cup', 'book', 'pencil', 'window']}, 'answerKey': 'A'}]

PIQA sample:
[{'goal': 'How to make coffee?', 'sol1': 'Pour hot water on coffee grounds.', 'sol2': 'Put the coffee in the freezer.', 'label': 0}, {'goal': 'How to clean a table?', 'sol1': 'Use a damp cloth and wipe it.', 'sol2': 'Use a hammer on it.', 'label': 0}]


In [2]:
# Inspection des champs de chaque dataset
print("CommonsenseQA example:")
print(csqa[0])
print("\nPIQA example:")
print(piqa[0])

CommonsenseQA example:
{'question': 'Where would you find a refrigerator?', 'choices': {'text': ['kitchen', 'forest', 'ocean', 'school']}, 'answerKey': 'A'}

PIQA example:
{'goal': 'How to make coffee?', 'sol1': 'Pour hot water on coffee grounds.', 'sol2': 'Put the coffee in the freezer.', 'label': 0}


In [3]:
import pandas as pd

rows = []
for item in csqa:
    rows.append({
        "dataset": "commonsense_qa",
        "question": item["question"],
        "choices": item["choices"]["text"],
        "answer_key": item.get("answerKey"),
    })

csqa_df = pd.DataFrame(rows)
print(csqa_df.head())

rows_piqa = []
for item in piqa:
    rows_piqa.append({
        "dataset": "piqa",
        "goal": item["goal"],
        "sol1": item["sol1"],
        "sol2": item["sol2"],
        "label": item.get("label"),
    })

piqa_df = pd.DataFrame(rows_piqa)
print(piqa_df.head())

          dataset                                 question  \
0  commonsense_qa     Where would you find a refrigerator?   
1  commonsense_qa  What do you usually use to drink water?   

                            choices answer_key  
0  [kitchen, forest, ocean, school]          A  
1       [cup, book, pencil, window]          A  
  dataset                   goal                               sol1  \
0    piqa    How to make coffee?  Pour hot water on coffee grounds.   
1    piqa  How to clean a table?      Use a damp cloth and wipe it.   

                             sol2  label  
0  Put the coffee in the freezer.      0  
1             Use a hammer on it.      0  


In [4]:
# Statistiques de base
print("Taille CSQA:", len(csqa_df))
print("Taille PIQA:", len(piqa_df))
print("Nombre moyen de choix CSQA:", csqa_df["choices"].apply(len).mean())
print("Nombre moyen de mots dans la question CSQA:", csqa_df["question"].str.split().str.len().mean())

Taille CSQA: 2
Taille PIQA: 2
Nombre moyen de choix CSQA: 4.0
Nombre moyen de mots dans la question CSQA: 7.0


### Interpretation

On observe que les datasets n’ont pas exactement le même format :
- CommonsenseQA contient plusieurs choix et une clé de réponse.
- PIQA présente un objectif et deux solutions candidates.

Cette différence justifie une normalisation légère avant l’évaluation.

In [5]:
# Distribution simple des longues questions et des choix
csqa_df["n_choices"] = csqa_df["choices"].apply(len)
print(csqa_df[["question", "n_choices"]].head())
print("Répartition de la taille des questions :")
print(csqa_df["question"].str.split().str.len().describe())


                                  question  n_choices
0     Where would you find a refrigerator?          4
1  What do you usually use to drink water?          4
Répartition de la taille des questions :
count    2.000000
mean     7.000000
std      1.414214
min      6.000000
25%      6.500000
50%      7.000000
75%      7.500000
max      8.000000
Name: question, dtype: float64
